In [44]:
## Load the names corpus and split into test set, dev test set and training set

In [45]:
import nltk, random
nltk.download('names')
from nltk.corpus import names

[nltk_data] Downloading package names to /Users/frazaslam/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [46]:
labeled_names = ([(n, "male") for n in names.words("male.txt")] +
                 [(n, "female") for n in names.words("female.txt")])

In [47]:
random.seed(1)
random.shuffle(labeled_names)

In [48]:
test_names = labeled_names[:500]
devtest_names = labeled_names[500:1000]
train_names = labeled_names[1000:]
print(len(test_names), len(devtest_names), len(train_names))

500 500 6944


In [49]:
## Starting Point - Guessing from the last letter

def gender_features(word):
    return {"last_letter": word[-1]}

print(gender_features("Fraz"))

{'last_letter': 'z'}


In [50]:
train_set = [(gender_features(n), g) for (n,g) in train_names]
devtest_set = [(gender_features(n), g) for (n,g) in devtest_names]

classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(classifier, devtest_set))
classifier.show_most_informative_features(5)

0.754
Most Informative Features
             last_letter = 'a'            female : male   =     33.2 : 1.0
             last_letter = 'k'              male : female =     30.4 : 1.0
             last_letter = 'p'              male : female =     17.3 : 1.0
             last_letter = 'f'              male : female =     16.4 : 1.0
             last_letter = 'v'              male : female =     16.2 : 1.0


In [51]:
## Look at mistakes and try single clues, error analysis

errors = []
for (name, tag) in devtest_names:
    guess = classifier.classify(gender_features(name))
    if guess != tag:
        errors.append((tag, guess, name))

print(len(errors))
for (tag, guess, name) in sorted(errors)[:30]:
    print("correct", tag, " guess", guess, " name:", name)

## There are alot of names that end in n that are being guessed as male but are female. ex. Allyson, Joslyn, Karen
## There are alot of names ending in s for females that are being guessed as male. ex. chriss, bliss, alyss, ardis

123
correct female  guess male  name: Allyson
correct female  guess male  name: Alyss
correct female  guess male  name: Ardis
correct female  guess male  name: Bird
correct female  guess male  name: Bliss
correct female  guess male  name: Cam
correct female  guess male  name: Caroljean
correct female  guess male  name: Carolynn
correct female  guess male  name: Chriss
correct female  guess male  name: Christan
correct female  guess male  name: Consuelo
correct female  guess male  name: Dallas
correct female  guess male  name: Dawn
correct female  guess male  name: Diahann
correct female  guess male  name: Diann
correct female  guess male  name: Dionis
correct female  guess male  name: Easter
correct female  guess male  name: Eden
correct female  guess male  name: Eilis
correct female  guess male  name: Ester
correct female  guess male  name: Fanchon
correct female  guess male  name: Gaynor
correct female  guess male  name: Gwendolen
correct female  guess male  name: Harriet
correct fem

In [52]:
def score(feature_function):
    train_set   = [(feature_function(n), g) for (n, g) in train_names]
    devtest_set = [(feature_function(n), g) for (n, g) in devtest_names]
    clf = nltk.NaiveBayesClassifier.train(train_set)
    return nltk.classify.accuracy(clf, devtest_set)

In [53]:
## Trying last 2-letters
def last_two(word):
    return{"last two": word[-2:]}
print(last_two("Fraz"))
print(score(last_two))

## this indicates a slight improvement in the score

{'last two': 'az'}
0.792


In [54]:
## Trying First Letter
## Decrease in score
def first_letter(word):
    return {"first_letter": word[0].lower()}

print(score(first_letter))


## Length of the name
## Decrease in score 

def length_name(word):
    return{"length_of_name": len(word)}

print(score(length_name))
    
## Last 3 letters
## Increase in score 

def last_three(word):
    return{"last_three_letters": word[-3:]}

print(score(last_three))


0.634
0.654
0.778


In [63]:
"""
So we tried different clues to improve our dev-test score and the end results were as such
Clue 	               Dev-test score
first letter	       0.634
length	               0.654
last letter 	       0.754
last three letters	   0.778
last two letters	   0.792

As we can see the greatest score is from observing the last two letters of a name. First letter and length are both
exceptionally weak. One reason why the last two letters outperforms the last three letters is potential overfitting
from the model with excess detail. 

"""

'\nSo we tried different clues to improve our dev-test score and the end results were as such\nClue \t               Dev-test score\nfirst letter\t       0.634\nlength\t               0.654\nlast letter \t       0.754\nlast three letters\t   0.778\nlast two letters\t   0.792\n\nAs we can see the greatest score is from observing the last two letters of a name. First letter and length are both\nexceptionally weak. One reason why the last two letters outperforms the last three letters is potential overfitting\nfrom the model with excess detail. \n\n'

In [56]:
## Combining Different Clues, trying multiple features and feature combinations

def combo_A(word):
    return {**gender_features(word), **last_two(word)}

print(combo_A("Joslyn"))
print(score(combo_A))



{'last_letter': 'n', 'last two': 'yn'}
0.782


In [57]:
def combo_B(word):
    return {**combo_A(word), **first_letter(word)}

print(combo_B("Joslyn"))
print(score(combo_B))

{'last_letter': 'n', 'last two': 'yn', 'first_letter': 'j'}
0.784


In [58]:
def combo_C(word):
    return {**combo_B(word), **length_name(word)}

print(combo_C("Joslyn"))
print(score(combo_C))

{'last_letter': 'n', 'last two': 'yn', 'first_letter': 'j', 'length_of_name': 6}
0.784


In [59]:
def combo_D(word):
    return {**combo_C(word), **last_three(word)}

print(combo_D("Joslyn"))
print(score(combo_D))

{'last_letter': 'n', 'last two': 'yn', 'first_letter': 'j', 'length_of_name': 6, 'last_three_letters': 'lyn'}
0.818


In [60]:
## Remove last letter and keep all other clues 
def combo_E(word):
    return {**last_two(word), **first_letter(word), **length_name(word), **last_three(word)}

print(combo_E("Joslyn"))
print(score(combo_E))


""" Removing a redudant clue here improved the score and returned a topscore, we removed last letter


Combo	Clues	                      Dev-test
A	    last letter + last two	      0.782
B	     A + first letter	          0.784
C	     B + length	                  0.784
D	     C + last three	              0.818
E	     D without last letter 	      0.828


"""



{'last two': 'yn', 'first_letter': 'j', 'length_of_name': 6, 'last_three_letters': 'lyn'}
0.828


' Removing a redudant clue here improved the score and returned a topscore, we removed last letter\n\n\nCombo\tClues\t                      Dev-test\nA\t    last letter + last two\t      0.782\nB\t     A + first letter\t          0.784\nC\t     B + length\t                  0.784\nD\t     C + last three\t              0.818\nE\t     D without last letter \t      0.828\n\n\n'

In [61]:
## Testing the Classifier on the Test Set

train_set = [(combo_E(n), g) for (n,g) in train_names]
test_set = [(combo_E(n), g) for (n,g) in test_names]
final_classifier = nltk.NaiveBayesClassifier.train(train_set)
print(nltk.classify.accuracy(final_classifier, test_set))

0.81


In [62]:
## Compare Dev-test and answer "is this expected?"

"""

Dev-Test - 0.828
Test - 0.810

Based on the test set we scored slightly lower by almost 2 points. Yes this is expected. The test score was slightly lower
than the dev-test score but does not warrant a concern. It is largely due to real-world differences betweeen the two sample sets.
There is a possibility of there being more similar names in the dev-test set than the test set. I would not mark is as a 
meaningful difference. Its an honest score with a new fresh sample set. 

"""

'\nDev-Test - 0.828\nTest - 0.810\n\nBased on the test set we scored slightly lower by almost 2 points. Yes this is expected. The test score was slightly lower\nthan the dev-test score but does not warrant a concern. It is largely due to real-world differences betweeen the two sample sets.\nThere is a possibility of there being more similar names in the dev-test set than the test set. I would not mark is as a \nmeaningful difference. Its an honest score with a new fresh sample set. \n'